### Imports

In [1]:
from pathlib import Path
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

PyTorch version : 2.14.0+cu126
CUDA available  : True


### Project Paths

In [2]:
current_directory = Path.cwd()
tokenizer_directory = current_directory / "tokenizer"
tokenizer_path = tokenizer_directory / "tokenizer.json"
tokenizer_config_path = tokenizer_directory / "tokenizer_config.json"
print("Current directory :", current_directory)
print("Tokenizer exists  :", tokenizer_path.exists())
print("Config exists     :", tokenizer_config_path.exists())

Current directory : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model
Tokenizer exists  : True
Config exists     : True


In [3]:
## Load tokenizer
with open(tokenizer_config_path, 'r', encoding='utf-8') as file:
    tokenizer_config = json.load(file)

vocab_size = tokenizer_config['vocab_size']
print("Vocabulary size :", vocab_size)
print("Algorithm       :", tokenizer_config["algorithm"])

Vocabulary size : 21000
Algorithm       : WordPiece


### Model Configuration

In [5]:
model_config = {
    "vocab_size": vocab_size,          # the vocabulary size that our tokenizer outputs
    "max_sequence_length": 256,        # also called "context window", the lenght of input sequence in one go

    "hidden_size": 128,                # the dimensionality of each token's representation throughout the Transfomer Backbone : [B, 256] to [B, 256, 218]
    "num_hidden_layers": 4,            # number of transformer encoder blocks

    "num_attention_heads": 4,          # multi head attention - 4 different perspectives
    "head_dimension": 32,              # 128 /4 = 32

    "intermediate_size": 256,          # expansion size - The purpose is to give the model a larger intermediate space in which to perfom nonlinear transformations

    "hidden_dropout": 0.05,            # randomly neurons will be switched off in both the FFN layer and Attention layer
    "attention_dropout": 0.05,

    "layer_norm_eps": 1e-5,            # the epsilon in the denominator

    "rope_theta": 10000.0,             # base for rotation of embedding pairs 
    "rotary_dimension": 32,            # all positions of the Q,K matrices will rotate

    "initializer_std": 0.02,           # random weight initializer
}

for key, value in model_config.items():
    print(f"{key:24s}: {value}")

vocab_size              : 21000
max_sequence_length     : 256
hidden_size             : 128
num_hidden_layers       : 4
num_attention_heads     : 4
head_dimension          : 32
intermediate_size       : 256
hidden_dropout          : 0.05
attention_dropout       : 0.05
layer_norm_eps          : 1e-05
rope_theta              : 10000.0
rotary_dimension        : 32
initializer_std         : 0.02


In [6]:
assert (
    model_config["hidden_size"]
    == model_config["num_attention_heads"]
    * model_config["head_dimension"]
)

assert model_config["rotary_dimension"] <= model_config["head_dimension"]

assert model_config["rotary_dimension"] % 2 == 0

print("Model configuration is valid.")

Model configuration is valid.


### Tokenization 

In [ ]:
## load tokenizer
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(str(tokenizer_path))           # loads a previously saved tokenizer, including its vocabulary, WordPiece model, normalization, pre-tokenization and other configurations
print("Tokenizer loaded.")
print("Vocabulary size :", tokenizer.get_vocab_size())

Tokenizer loaded.
Vocabulary size : 21000


In [8]:
## check tokenizer
sample_text = "The patient has patellofemoral chondromalacia."

encoding = tokenizer.encode(sample_text)

print("Text   :", sample_text)
print("Tokens :", encoding.tokens)
print("IDs    :", encoding.ids)

Text   : The patient has patellofemoral chondromalacia.
Tokens : ['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']
IDs    : [1970, 14284, 2186, 3675, 6781, 11885, 20391, 1936, 5521, 1930, 2003, 2804, 8046, 18]


### Embedding Layer

In [17]:
class TokenEmbedding(nn.Module):                                 # oops inheritance at work
    def __init__(self, vocab_size, hidden_size, dropout):        # defines what the layer contains
        super().__init__()

        self.token_embedding = nn.Embedding(                     # creates a learnable embedding matrix for each token in the vocab
            num_embeddings=vocab_size,
            embedding_dim=hidden_size                    
        )

        self.dropout = nn.Dropout(dropout)                       # creates a dropout layer - acts as regularization and prevents overfitting

    def forward(self, input_ids):

        embeddings = self.token_embedding(input_ids)

        embeddings = self.dropout(embeddings)

        return embeddings

In [18]:
## test token embeddings
input_ids = torch.tensor(                       # need to convert the ids to tensor
    [encoding.ids],                         # from the test run above
    dtype = torch.long
)

token_embedding = TokenEmbedding(                      # initializing our embedding object with the configurations we had set earlier
    vocab_size = model_config['vocab_size'],
    hidden_size =  model_config['hidden_size'],
    dropout = model_config['hidden_dropout']
)

embedding_output = token_embedding(input_ids)          # passing the input through the object

print("Tokens           :", encoding.tokens)
print("Input IDs shape  :", input_ids.shape)
print("Embedding shape  :", embedding_output.shape)

Tokens           : ['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']
Input IDs shape  : torch.Size([1, 14])
Embedding shape  : torch.Size([1, 14, 128])


In [12]:
torch.tensor(encoding.ids)

tensor([ 1970, 14284,  2186,  3675,  6781, 11885, 20391,  1936,  5521,  1930,
         2003,  2804,  8046,    18])

# Transformer Block

### Pre - Layer Norm

In [ ]:
pre_attention_layer_norm = nn.LayerNorm(                    # pytorch's layernorm module. Layer norm acts on each token separately
    normalized_shape = model_config['hidden_size'],
    eps = model_config['layer_norm_eps']
)

normalized_embeddings = pre_attention_layer_norm(           # passing our embedded input through the layernorm
    embedding_output
)

print("Input shape          :", embedding_output.shape)
print("Normalized shape     :", normalized_embeddings.shape) # confirming if shapes remain same or not
                                                             # for every token, normalizes across its 128 hidden dimensions

Input shape          : torch.Size([1, 14, 128])
Normalized shape     : torch.Size([1, 14, 128])


### Query (Q) - Key (K) - Value(V) vectors

In [22]:
query_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

key_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

value_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

query = query_projection(normalized_embeddings)
key = key_projection(normalized_embeddings)
value = value_projection(normalized_embeddings)

print("Input shape :", normalized_embeddings.shape)
print("Query shape :", query.shape)
print("Key shape   :", key.shape)
print("Value shape :", value.shape)

Input shape : torch.Size([1, 14, 128])
Query shape : torch.Size([1, 14, 128])
Key shape   : torch.Size([1, 14, 128])
Value shape : torch.Size([1, 14, 128])


In [23]:
## ^^ Mathematically ----- Q = XW_q, K = XW_k, V = XW_v

In [ ]:
## inspect projection weights
print("Query weight shape :", query_projection.weight.shape)
print("Key weight shape   :", key_projection.weight.shape)
print("Value weight shape :", value_projection.weight.shape)

print("\nQuery bias :", query_projection.bias)
print("Key bias   :", key_projection.bias)
print("Value bias :", value_projection.bias)

Query weight shape : torch.Size([128, 128])
Key weight shape   : torch.Size([128, 128])
Value weight shape : torch.Size([128, 128])

Query bias : None
Key bias   : None
Value bias : None


### Multi Head split

In [27]:
## this cell will reshape our Q, K, V tensors into multiple attention heads

batch_size, sequence_length, hidden_size = query.shape

num_heads = model_config['num_attention_heads']
head_dimension = model_config['head_dimension']

query = query.view(                                # reshapes query from 3 dimensions to 4 dimensions
    batch_size, 
    sequence_length,
    num_heads,                                     # this is the new dimension we are adding, essentially inserting 4 matrices instead of the 1 currently
    head_dimension                                 # instead of hidden_size, now each head will have dimension hidden_size/4
)

key = key.view( 
    batch_size,
    sequence_length,
    num_heads,
    head_dimension
)

value = value.view(
    batch_size,
    sequence_length,
    num_heads,
    head_dimension
)

print("Query shape :", query.shape)
print("Key shape   :", key.shape)
print("Value shape :", value.shape)

Query shape : torch.Size([1, 14, 4, 32])
Key shape   : torch.Size([1, 14, 4, 32])
Value shape : torch.Size([1, 14, 4, 32])
